# Interrupted Time Series: Uber Platform-Wide Marketing Campaign Impact

## Business Problem

Uber launched a major TV + digital marketing campaign to boost ride revenue across the entire platform. The campaign went live on **day 90** of our observation window. We have **180 days** of daily ride revenue data.

**Causal question**: Did the marketing campaign cause a sustained increase in daily ride revenue, and if so, how large was the immediate lift and the change in growth trajectory?

Because the campaign was platform-wide (every market received it simultaneously), there is no natural control group. This makes ITS the right tool: we model the pre-campaign trend and test whether the post-campaign data deviates from that extrapolated trend.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Simulate Daily Ride Revenue

We generate 180 days of data with:
- **Pre-existing upward trend**: Revenue was already growing at ~$500/day before the campaign
- **Day-of-week seasonality**: Weekends have higher revenue (Fri/Sat ~$8K above baseline)
- **Campaign level shift**: Immediate +$15,000/day jump when the campaign starts
- **Campaign slope change**: Additional +$200/day growth rate on top of the pre-existing trend
- **Noise**: Real-world variability

In [ ]:
n_days = 180
intervention_day = 90

time = np.arange(1, n_days + 1)
post = (time >= intervention_day).astype(int)
time_since = np.maximum(time - intervention_day, 0)

day_of_week = time % 7  # 0=Mon ... 6=Sun

# Seasonality: Fri(4) and Sat(5) boost, Sun(6) moderate boost, mid-week dip
seasonality = np.where(day_of_week == 4, 6000,
              np.where(day_of_week == 5, 8000,
              np.where(day_of_week == 6, 4000,
              np.where(day_of_week == 0, -2000,
              np.where(day_of_week == 1, -3000,
              np.where(day_of_week == 2, -1000, 0))))))

# True DGP
baseline = 200_000
pre_trend = 500  # $/day organic growth
true_level_shift = 15_000
true_slope_change = 200  # additional $/day after campaign

noise = np.random.normal(0, 5000, n_days)

revenue = (
    baseline
    + pre_trend * time
    + true_level_shift * post
    + true_slope_change * time_since
    + seasonality
    + noise
)

df = pd.DataFrame({
    'day': time,
    'revenue': revenue,
    'post': post,
    'time_since': time_since,
    'day_of_week': day_of_week
})

# Create day-of-week dummies
dow_dummies = pd.get_dummies(df['day_of_week'], prefix='dow', drop_first=True)
df = pd.concat([df, dow_dummies], axis=1)

print(f"Days: {n_days}, Intervention at day: {intervention_day}")
print(f"True level shift: ${true_level_shift:,}/day")
print(f"True slope change: ${true_slope_change:,}/day")
df.head(10)

## 2. Naive Before/After Comparison (Why It Fails)

The simplest approach: compare mean revenue before vs. after the campaign. This is **wrong** because it ignores the pre-existing upward trend. Revenue was already growing at $500/day, so part of the post-campaign increase would have happened anyway.

In [ ]:
pre_mean = df.loc[df['post'] == 0, 'revenue'].mean()
post_mean = df.loc[df['post'] == 1, 'revenue'].mean()
naive_diff = post_mean - pre_mean

# What naive gets wrong: the pre-trend alone would have raised the mean
# Average day in pre-period: ~45, average day in post-period: ~135
# Pre-trend contribution to naive difference: 500 * (135 - 45) = $45,000
expected_from_trend = pre_trend * (135 - 45)

print(f"Pre-campaign mean revenue:  ${pre_mean:,.0f}")
print(f"Post-campaign mean revenue: ${post_mean:,.0f}")
print(f"Naive difference:           ${naive_diff:,.0f}")
print(f"\nBut ${expected_from_trend:,} of this is explained by the pre-existing trend alone.")
print(f"True campaign level shift is only ${true_level_shift:,}/day.")
print(f"\nNaive estimate is massively biased upward because it confounds")
print(f"the organic growth trajectory with the campaign effect.")

## 3. Why ITS and Not Other Methods?

| Method | Why it doesn't work here |
|--------|-------------------------|
| **A/B Test** | Impossible — the campaign is platform-wide (TV, billboards, digital). You can't randomize who sees a Super Bowl ad. |
| **Difference-in-Differences** | Requires a control group that didn't receive treatment. All Uber markets got the campaign simultaneously. No untreated group exists. |
| **Synthetic Control** | Requires a pool of untreated "donor" units. Uber is a single platform — there's no collection of similar ride-sharing platforms to construct a synthetic Uber. |
| **Propensity Score Matching** | Designed for individual-level treatment assignment. Here the treatment is at the platform level — every user/market is treated. There are no untreated individuals to match. |
| **Regression Discontinuity** | Requires treatment assigned by a continuous running variable crossing a threshold. There's no such threshold here — the campaign is time-based, not score-based. |

**ITS is the right fit because**: We have one unit (the Uber platform), a long time series (180 daily observations), a clear intervention date, and no control group. ITS leverages the pre-intervention trajectory as its own "control" by extrapolating what would have happened without the campaign.

## 4. Visualize the Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.scatter(df['day'], df['revenue'], c=df['post'].map({0: 'steelblue', 1: 'coral'}),
           alpha=0.5, s=20, label='_nolegend_')
ax.axvline(x=intervention_day, color='red', linestyle='--', linewidth=2, label='Campaign start')

# Pre-trend extrapolation (counterfactual)
counterfactual = baseline + pre_trend * time
ax.plot(time, counterfactual, 'k--', alpha=0.4, label='Pre-trend extrapolation (counterfactual)')

ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Daily Revenue ($)', fontsize=12)
ax.set_title('Uber Daily Ride Revenue — Marketing Campaign ITS Analysis', fontsize=14)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 5. Segmented Regression — The Core ITS Model

The standard ITS regression is:

$$Y_t = \beta_0 + \beta_1 \cdot time_t + \beta_2 \cdot post_t + \beta_3 \cdot time\_since_t + \epsilon_t$$

**Why each term is needed:**

| Term | Coefficient | What it captures | Why it's needed |
|------|-------------|------------------|-----------------|
| `time` | $\beta_1$ | Pre-intervention trend (organic growth) | Without it, any pre-existing growth gets falsely attributed to the campaign |
| `post` | $\beta_2$ | **Immediate level shift** at intervention | Captures the sudden jump in revenue the moment the campaign starts |
| `time_since` | $\beta_3$ | **Change in slope** after intervention | Captures whether the campaign changed the growth *trajectory*, not just the level |

If we omitted `time`, $\beta_2$ would absorb the pre-existing trend → **upward bias**.  
If we omitted `time_since`, we'd miss that the campaign might have accelerated (or decelerated) growth.

In [ ]:
model_basic = smf.ols('revenue ~ day + post + time_since', data=df).fit()
print(model_basic.summary())

print("\n" + "="*60)
print("COEFFICIENT INTERPRETATION")
print("="*60)
print(f"Pre-trend (day):           ${model_basic.params['day']:,.0f}/day")
print(f"  → True value: ${pre_trend:,}/day")
print(f"Level shift (post):        ${model_basic.params['post']:,.0f}")
print(f"  → True value: ${true_level_shift:,}")
print(f"Slope change (time_since): ${model_basic.params['time_since']:,.0f}/day")
print(f"  → True value: ${true_slope_change:,}/day")

## 6. Add Seasonality Controls

**Why this matters**: Daily revenue has strong day-of-week patterns (weekends are higher). If the campaign happened to start on a particular day of the week, uncontrolled seasonality could bias the level-shift estimate. Even if the start day is "average," seasonality inflates residual variance and reduces statistical power.

Adding day-of-week dummies absorbs this cyclical variation, yielding:
- More precise estimates (tighter confidence intervals)
- Less risk of spurious significance from seasonal alignment

In [ ]:
dow_cols = [c for c in df.columns if c.startswith('dow_')]
formula_seasonal = 'revenue ~ day + post + time_since + ' + ' + '.join(dow_cols)

model_seasonal = smf.ols(formula_seasonal, data=df).fit()
print(model_seasonal.summary())

print("\n" + "="*60)
print("COMPARISON: Basic vs. Seasonality-Adjusted")
print("="*60)
for param in ['day', 'post', 'time_since']:
    basic_se = model_basic.bse[param]
    seasonal_se = model_seasonal.bse[param]
    reduction = (1 - seasonal_se / basic_se) * 100
    print(f"{param:15s}: SE {basic_se:>8,.0f} → {seasonal_se:>8,.0f}  ({reduction:+.1f}% change)")

print(f"\nR² improved from {model_basic.rsquared:.4f} to {model_seasonal.rsquared:.4f}")
print("Seasonality controls absorb day-of-week variance → sharper estimates.")

## 7. HAC Standard Errors (Newey-West)

**Why this is critical**: Time series data almost always has autocorrelation — today's revenue is correlated with yesterday's. OLS assumes independent errors, so standard errors from OLS are too small → we'd over-reject the null and get false confidence.

**Heteroskedasticity and Autocorrelation Consistent (HAC)** standard errors (Newey-West) correct for this. We first test for autocorrelation, then re-estimate with robust SEs.

In [ ]:
# Test for autocorrelation in residuals
bg_test = acorr_breusch_godfrey(model_seasonal, nlags=7)
print("Breusch-Godfrey test for autocorrelation (7 lags):")
print(f"  LM statistic: {bg_test[0]:.2f}")
print(f"  p-value:      {bg_test[1]:.4f}")
if bg_test[1] < 0.05:
    print("  → Significant autocorrelation detected. HAC SEs are necessary.")
else:
    print("  → No significant autocorrelation at 5% level (HAC still recommended as insurance).")

print("\n" + "="*60)

# Re-fit with HAC (Newey-West) standard errors
# Rule of thumb for bandwidth: 0.75 * T^(1/3)
nw_lags = int(0.75 * len(df) ** (1/3))
print(f"Using Newey-West with {nw_lags} lags\n")

model_hac = smf.ols(formula_seasonal, data=df).fit(
    cov_type='HAC', cov_kwds={'maxlags': nw_lags}
)

print("KEY ESTIMATES WITH HAC STANDARD ERRORS")
print("="*60)
for param in ['day', 'post', 'time_since']:
    coef = model_hac.params[param]
    se_ols = model_seasonal.bse[param]
    se_hac = model_hac.bse[param]
    pval = model_hac.pvalues[param]
    print(f"{param:15s}: coef={coef:>10,.0f}  SE(OLS)={se_ols:>7,.0f}  SE(HAC)={se_hac:>7,.0f}  p={pval:.4f}")

print(f"\nNote: HAC SEs are typically larger than OLS SEs. If a result")
print(f"was only 'barely significant' under OLS, it may lose significance under HAC.")

## 8. Placebo Test at Day 45

**Logic**: If our ITS model is valid, a fake intervention at day 45 (well before the real campaign) should show **no** significant level shift or slope change. Finding a significant effect at a placebo date would suggest the model is picking up pre-existing structural breaks, not the campaign.

In [ ]:
placebo_day = 45

# Use only pre-intervention data so the real campaign can't contaminate
df_pre = df[df['day'] < intervention_day].copy()
df_pre['placebo_post'] = (df_pre['day'] >= placebo_day).astype(int)
df_pre['placebo_time_since'] = np.maximum(df_pre['day'] - placebo_day, 0)

dow_cols_pre = [c for c in df_pre.columns if c.startswith('dow_')]
formula_placebo = 'revenue ~ day + placebo_post + placebo_time_since + ' + ' + '.join(dow_cols_pre)

model_placebo = smf.ols(formula_placebo, data=df_pre).fit(
    cov_type='HAC', cov_kwds={'maxlags': nw_lags}
)

print("PLACEBO TEST — Fake intervention at day 45")
print("="*60)
for param in ['placebo_post', 'placebo_time_since']:
    coef = model_placebo.params[param]
    pval = model_placebo.pvalues[param]
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
    print(f"{param:25s}: coef={coef:>10,.0f}  p={pval:.4f} {sig}")

placebo_p = model_placebo.pvalues['placebo_post']
if placebo_p > 0.05:
    print(f"\n✓ Placebo level shift is NOT significant (p={placebo_p:.3f}).")
    print("  This supports the validity of our ITS design.")
else:
    print(f"\n✗ Placebo level shift IS significant (p={placebo_p:.3f}).")
    print("  This is concerning — there may be a pre-existing structural break.")

## 9. Sensitivity: Varying the Pre/Post Window

ITS estimates can be sensitive to how much data we include. If the pre-trend isn't truly linear over the full window, using a shorter window may give different results. We check robustness by varying the symmetric window around the intervention.

In [ ]:
windows = [30, 45, 60, 75, 89]  # days before AND after intervention to include
results = []

for w in windows:
    df_w = df[(df['day'] >= intervention_day - w) & (df['day'] < intervention_day + w)].copy()
    dow_w = [c for c in df_w.columns if c.startswith('dow_')]
    formula_w = 'revenue ~ day + post + time_since + ' + ' + '.join(dow_w)
    m = smf.ols(formula_w, data=df_w).fit(
        cov_type='HAC', cov_kwds={'maxlags': max(1, int(0.75 * len(df_w) ** (1/3)))}
    )
    results.append({
        'window': f'±{w} days',
        'n_obs': len(df_w),
        'level_shift': m.params['post'],
        'level_se': m.bse['post'],
        'level_p': m.pvalues['post'],
        'slope_change': m.params['time_since'],
        'slope_se': m.bse['time_since'],
        'slope_p': m.pvalues['time_since']
    })

sensitivity = pd.DataFrame(results)
print("SENSITIVITY ANALYSIS: Varying Pre/Post Window")
print("="*80)
print(f"True level shift: ${true_level_shift:,}  |  True slope change: ${true_slope_change:,}/day")
print("-"*80)
for _, row in sensitivity.iterrows():
    print(f"{row['window']:>10s} (n={row['n_obs']:3d}): "
          f"level=${row['level_shift']:>8,.0f} (p={row['level_p']:.3f})  "
          f"slope=${row['slope_change']:>6,.0f} (p={row['slope_p']:.3f})")

print("\nResults should be stable across windows if the pre-trend is well-specified.")
print("Large variation would suggest model misspecification (e.g., non-linear pre-trend).")

## 10. What Happens With a Co-Occurring Event (Confound)

The core ITS assumption is that **nothing else changed at the intervention point**. Here we simulate a competitor exiting the market on the same day as the campaign. This confound inflates the level-shift estimate because ITS cannot separate the two effects.

This demonstrates **the most dangerous threat to ITS validity**: co-occurring events.

In [ ]:
# Simulate: a competitor exits on the same day, adding $10K/day to Uber's revenue
confound_effect = 10_000
df_confound = df.copy()
df_confound['revenue'] = df_confound['revenue'] + confound_effect * df_confound['post']

dow_cols_c = [c for c in df_confound.columns if c.startswith('dow_')]
formula_c = 'revenue ~ day + post + time_since + ' + ' + '.join(dow_cols_c)

model_confound = smf.ols(formula_c, data=df_confound).fit(
    cov_type='HAC', cov_kwds={'maxlags': nw_lags}
)

print("CONFOUNDED ITS — Competitor exits market on same day as campaign")
print("="*60)
print(f"True campaign level shift:  ${true_level_shift:,}")
print(f"True competitor effect:     ${confound_effect:,}")
print(f"Combined true effect:       ${true_level_shift + confound_effect:,}")
print(f"Estimated level shift:      ${model_confound.params['post']:,.0f}")
print(f"\nITS CANNOT separate these two effects.")
print(f"The estimate absorbs both the campaign AND the competitor exit.")
print(f"\nThis is why domain knowledge and event logs are critical:")
print(f"you must verify that no other major event coincided with the intervention.")
print(f"\nMitigations:")
print(f"  - Maintain a log of all major events/launches/competitor moves")
print(f"  - Use Controlled ITS (CITS) if any comparison group exists")
print(f"  - Check outcome series in related-but-unaffected metrics")

## 11. Visualization — Fitted Model with Counterfactual

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Observed data
ax.scatter(df['day'], df['revenue'], c=df['post'].map({0: 'steelblue', 1: 'coral'}),
           alpha=0.4, s=15, zorder=2)

# Fitted values from seasonal model
df['fitted'] = model_seasonal.fittedvalues

# Counterfactual: what would have happened without the campaign
df_counter = df.copy()
df_counter['post'] = 0
df_counter['time_since'] = 0
df['counterfactual'] = model_seasonal.predict(df_counter)

ax.plot(df['day'], df['fitted'], 'darkred', linewidth=2, label='Fitted (with campaign)', zorder=3)
ax.plot(df['day'], df['counterfactual'], 'k--', linewidth=2, alpha=0.6,
        label='Counterfactual (no campaign)', zorder=3)

# Shade the treatment effect
post_mask = df['post'] == 1
ax.fill_between(df.loc[post_mask, 'day'],
                df.loc[post_mask, 'counterfactual'],
                df.loc[post_mask, 'fitted'],
                alpha=0.2, color='coral', label='Estimated campaign effect')

ax.axvline(x=intervention_day, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Day', fontsize=12)
ax.set_ylabel('Daily Revenue ($)', fontsize=12)
ax.set_title('ITS: Fitted Model vs. Counterfactual', fontsize=14)
ax.legend(fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 12. Key Takeaways

### What We Found
1. **The marketing campaign caused a statistically significant increase in daily revenue**, with an immediate level shift near the true $15,000/day and an additional growth acceleration near $200/day.
2. **Naive before/after comparison massively overestimates** the effect by confounding the pre-existing organic growth with the campaign impact.

### Method Lessons
3. **Segmented regression is the core ITS tool**: The `time`, `post`, and `time_since` terms each serve a distinct purpose. Omitting any one biases the others.
4. **Seasonality controls matter**: Day-of-week dummies reduced standard errors substantially and prevent seasonal aliasing.
5. **HAC standard errors are necessary**: Time series residuals are autocorrelated; OLS SEs are too optimistic. Always use Newey-West or similar.
6. **Placebo tests validate the design**: No false positive at day 45 supports the claim that our model isn't just picking up noise or structural breaks.
7. **Co-occurring events are the biggest threat**: ITS cannot separate simultaneous causes. Domain knowledge and event logging are essential complements.

### When to Move Beyond ITS
- If a comparison group becomes available → **Controlled ITS (CITS)** or **DiD**
- If donor time series exist → **Synthetic Control** or **CausalImpact (Bayesian Structural Time Series)**
- If the intervention is staggered across units → **Staggered DiD**